# 8. Revisione agentica di un contratto

Analizza clausole, importi e firme con il workflow agentic.

## 1. Import e configurazione

Carica librerie, variabili di ambiente, schema e documento.

In [ ]:
import json
import os
from pathlib import Path
from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.ai.contentunderstanding.models import ContentAnalyzer
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

In [ ]:
load_dotenv(r"..\.env", override=True)
document_bytes = Path(r"..\file\esercizio8.pdf").read_bytes()
definizione = json.loads(
    Path(r"..\modelli\contratto_agentic.json").read_text(encoding="utf-8")
)

## 2. Analisi agentica

Crea l'analyzer agentic, analizza il contratto e rimuove l'analyzer.

In [ ]:
with DefaultAzureCredential() as credential, ContentUnderstandingClient(
    endpoint=os.environ["FOUNDRY_SERVICES_ENDPOINT"], credential=credential,
) as client:
    client.begin_create_analyzer(
        analyzer_id="contratto_agentic",
        resource=ContentAnalyzer(definizione),
        allow_replace=True,
    ).result()
    try:
        result = client.begin_analyze_binary(
            analyzer_id="contratto_agentic",
            binary_input=document_bytes,
        ).result()
    finally:
        client.delete_analyzer(analyzer_id="contratto_agentic")

## 3. Risultato

Mostra calcoli, verifiche contrattuali ed esito motivato.

In [ ]:
fields = result.as_dict()["contents"][0]["fields"]
print(json.dumps(fields, ensure_ascii=False, indent=2))